Copyright (C) 2026 Michael Nowotny

This program is free software; you can redistribute it and/or modify
it under the terms of the GNU General Public License version 2 as
published by the Free Software Foundation.

This program is distributed in the hope that it will be useful,
but WITHOUT ANY WARRANTY; without even the implied warranty of
MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE. See the
GNU General Public License for more details.

# The Wells of Bangladesh

*Logistic regression, life-or-death decisions, and the power of uncertainty*

## Poisoned Water

In the 1990s, one of the largest mass poisonings in history was unfolding
in Bangladesh. Millions of tube wells -- drilled in the 1970s to provide
clean water and prevent cholera -- were contaminated with naturally occurring
**arsenic**. The wells were slowly poisoning the people they were meant
to save.

The government tested wells and marked the dangerous ones. Families now
faced a choice: **keep drinking from their poisoned well** (it was close,
it was familiar) **or walk to a safe well** that might be far away.

What determines whether a family switches? How far is the nearest safe
well? How educated is the household? How dangerous is the arsenic level
in their current well? These are questions with real consequences --
information campaigns targeting the right factors could save lives.

This dataset, from Gelman & Hill (2006), records the decisions of
**3,020 households** in Araihazar, Bangladesh. Each row is a family.
Each row is a life-or-death decision.

---

**What you will learn:**
- How the logit link function connects linear models to binary outcomes
- How to build logistic regression models progressively
- How to interpret posterior distributions of log-odds
- How to compare models using DIC and LOO
- How Bayesian uncertainty quantification leads to better decisions

In [ ]:
import os

import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import pyjags

## The Data: 3,020 Families, One Decision

For each household, we know:
- **switch**: did the family switch to a safe well? (yes/no)
- **arsenic**: arsenic level in the household's current well (multiples
  of the safe threshold -- so 2.0 means twice the safe level)
- **distance**: distance to the nearest safe well (meters)
- **education**: years of education of the head of household
- **association**: whether the household is a member of a community
  organization

In [ ]:
df = pd.read_csv(os.path.join("data", "wells.csv"), index_col=0)

# Convert switch to binary
df["y"] = (df["switch"] == "yes").astype(int)

# Scale distance to 100m units for numerical stability
df["dist100"] = df["distance"] / 100

print(f"Households:  {len(df)}")
print(f"Switched:    {df['y'].sum()} ({df['y'].mean():.0%})")
print(f"Did not:     {(1 - df['y']).sum():.0f} ({1 - df['y'].mean():.0%})")
df[["y", "arsenic", "dist100", "education"]].describe().round(2)

About 58% of families switched. The average arsenic level is 1.66
times the safe threshold. The average distance to a safe well is about
48 meters. Let's visualize the key relationships:

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Distance vs. switching
for val, color, label in [(1, "darkorange", "Switched"), (0, "steelblue", "Stayed")]:
    subset = df[df["y"] == val]
    axes[0].hist(subset["distance"], bins=30, alpha=0.5, color=color,
                 label=label, density=True)
axes[0].set_xlabel("Distance to safe well (m)")
axes[0].set_title("Closer wells → more switching")
axes[0].legend()

# Arsenic vs. switching
for val, color, label in [(1, "darkorange", "Switched"), (0, "steelblue", "Stayed")]:
    subset = df[df["y"] == val]
    axes[1].hist(subset["arsenic"], bins=30, alpha=0.5, color=color,
                 label=label, density=True, range=(0, 6))
axes[1].set_xlabel("Arsenic level (multiples of safe)")
axes[1].set_title("Higher arsenic → more switching")
axes[1].legend()

# Education vs. switching
for val, color, label in [(1, "darkorange", "Switched"), (0, "steelblue", "Stayed")]:
    subset = df[df["y"] == val]
    axes[2].hist(subset["education"], bins=15, alpha=0.5, color=color,
                 label=label, density=True)
axes[2].set_xlabel("Years of education")
axes[2].set_title("More education → more switching")
axes[2].legend()

plt.tight_layout()
plt.show()

The patterns make intuitive sense:
- Families **closer** to a safe well are more likely to switch (lower cost)
- Families with **higher arsenic** are more likely to switch (higher
  motivation)
- Families with **more education** are more likely to switch (better
  information)

Each of these insights could inform a public health intervention. But to
quantify them precisely -- and to understand their **joint** effect --
we need a model.

## The Logistic Model

In 1944, the biostatistician Joseph Berkson coined the word **logit** -- a
contraction of "logistic unit" -- to describe a transformation that maps
any real number to a probability between 0 and 1. The logistic function,
$p = 1/(1 + e^{-x})$, creates the S-shaped curve that rises steeply
through a threshold and then levels off.

Our model says:

$$\text{logit}(p_i) = \log\frac{p_i}{1 - p_i} = \alpha + \beta_1 \cdot \text{dist100}_i + \beta_2 \cdot \text{arsenic}_i + \beta_3 \cdot \text{education}_i$$

where $p_i$ is the probability that household $i$ switches wells.

In BUGS, this translates directly:

In [ ]:
model_code = """
model {
    for (i in 1:N) {
        y[i] ~ dbern(p[i])
        logit(p[i]) = alpha
                     + beta_dist * dist100[i]
                     + beta_arsenic * arsenic[i]
                     + beta_educ * education[i]
    }

    # Weakly informative priors
    alpha ~ dnorm(0, 1.0/25)
    beta_dist ~ dnorm(0, 1.0/25)
    beta_arsenic ~ dnorm(0, 1.0/25)
    beta_educ ~ dnorm(0, 1.0/25)
}
"""

jags_data = dict(
    y=df["y"].values,
    dist100=df["dist100"].values,
    arsenic=df["arsenic"].values,
    education=df["education"].values,
    N=len(df),
)

In [ ]:
model = pyjags.Model(
    code=model_code, data=jags_data,
    chains=4, adapt=1000, progress_bar=False, seed=42,
)
model.sample(1000, vars=[])  # burn-in
samples = model.sample(5000, vars=["alpha", "beta_dist", "beta_arsenic", "beta_educ"])

idata = pyjags.from_pyjags(samples)
az.summary(idata)

## Reading the Results

All R-hat values are 1.00. Let's look at the posterior distributions:

In [ ]:
az.plot_trace(idata)
plt.tight_layout()
plt.show()

In [ ]:
az.plot_forest(idata, var_names=["beta_dist", "beta_arsenic", "beta_educ"],
               combined=True)
plt.title("What drives the decision to switch wells?")
plt.tight_layout()
plt.show()

### Interpreting the Coefficients

These are **log-odds ratios**. Let's translate them into something
a public health official can act on:

In [ ]:
# Posterior means
beta_dist = idata["posterior"]["beta_dist"].values.flatten()
beta_arsenic = idata["posterior"]["beta_arsenic"].values.flatten()
beta_educ = idata["posterior"]["beta_educ"].values.flatten()

print("Odds ratios (multiplicative change in odds per unit increase):")
print(f"  Distance (+100m): {np.exp(beta_dist.mean()):.2f}x "
      f"  [{np.exp(np.percentile(beta_dist, 3)):.2f}, {np.exp(np.percentile(beta_dist, 97)):.2f}]")
print(f"  Arsenic (+1 unit): {np.exp(beta_arsenic.mean()):.2f}x "
      f"  [{np.exp(np.percentile(beta_arsenic, 3)):.2f}, {np.exp(np.percentile(beta_arsenic, 97)):.2f}]")
print(f"  Education (+1 yr): {np.exp(beta_educ.mean()):.2f}x "
      f"  [{np.exp(np.percentile(beta_educ, 3)):.2f}, {np.exp(np.percentile(beta_educ, 97)):.2f}]")
print()
print("In plain language:")
print(f"  Every 100m farther reduces the odds of switching by ~{(1-np.exp(beta_dist.mean()))*100:.0f}%")
print(f"  Doubling the arsenic level increases the odds by ~{(np.exp(beta_arsenic.mean())-1)*100:.0f}%")
print(f"  Each additional year of education increases the odds by ~{(np.exp(beta_educ.mean())-1)*100:.0f}%")

These findings have direct **policy implications**:

1. **Distance matters most.** Reducing the distance to safe wells --
   by drilling new ones in underserved areas -- would have the largest
   impact on switching behavior.

2. **Information matters.** Education increases switching probability,
   suggesting that **awareness campaigns** about arsenic risks would help.

3. **Arsenic level drives urgency.** Families with higher contamination
   are already more motivated -- they need logistics (closer safe wells),
   not more motivation.

None of these conclusions come with false certainty. The Bayesian
posterior gives us a **distribution** of plausible effect sizes for each
factor, not a single point estimate. The credible intervals above tell
policymakers how confident they should be.

## Visualizing the Decision Boundary

Let's see the model's predictions in action. For a family at the average
arsenic level, how does the probability of switching change with distance
and education?

In [ ]:
from scipy.special import expit  # logistic function

alpha_post = idata["posterior"]["alpha"].values.flatten()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Probability vs. distance at mean arsenic, for different education levels
dist_grid = np.linspace(0, 3, 200)  # 0 to 300m
mean_arsenic = df["arsenic"].mean()
for edu, color in [(0, "crimson"), (5, "darkorange"), (12, "steelblue")]:
    logit_p = alpha_post.mean() + beta_dist.mean() * dist_grid + \
              beta_arsenic.mean() * mean_arsenic + beta_educ.mean() * edu
    axes[0].plot(dist_grid * 100, expit(logit_p), color=color, linewidth=2,
                 label=f"{edu} years education")
axes[0].set_xlabel("Distance to safe well (m)")
axes[0].set_ylabel("Probability of switching")
axes[0].set_title(f"Effect of distance (arsenic = {mean_arsenic:.1f}x safe)")
axes[0].legend()
axes[0].set_ylim(0, 1)

# Probability vs. arsenic at mean distance, for different education levels
arsenic_grid = np.linspace(0.5, 6, 200)
mean_dist = df["dist100"].mean()
for edu, color in [(0, "crimson"), (5, "darkorange"), (12, "steelblue")]:
    logit_p = alpha_post.mean() + beta_dist.mean() * mean_dist + \
              beta_arsenic.mean() * arsenic_grid + beta_educ.mean() * edu
    axes[1].plot(arsenic_grid, expit(logit_p), color=color, linewidth=2,
                 label=f"{edu} years education")
axes[1].set_xlabel("Arsenic level (multiples of safe)")
axes[1].set_ylabel("Probability of switching")
axes[1].set_title(f"Effect of arsenic (distance = {mean_dist*100:.0f}m)")
axes[1].legend()
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.show()

The S-shaped logistic curve in action. Education shifts the entire curve
upward -- at every distance, at every arsenic level, more educated
families are more likely to switch. This is the **invisible threshold**
of the logistic function: a smooth transition from "probably won't
switch" to "probably will," shaped by the interplay of all three factors.

## What Comes Next

We used Bayesian logistic regression to understand a real public health
crisis. Every coefficient in our model has a tangible meaning and a
credible interval that honestly represents our uncertainty.

In the next notebook, we enter the world of **quantitative finance**.
Every time a stock trades, there is a hidden cost -- the *bid-ask spread*
-- that you can't observe directly in the data. Estimating it requires a
model with **latent variables** (the unobserved efficient price) and
**discrete parameters** (was each trade a buy or a sell?). This is
exactly the kind of model where JAGS excels: its Gibbs sampler can draw
discrete trade directions directly, something that gradient-based
samplers like NUTS cannot do.

---

**Further reading:**
- Gelman, A. & Hill, J. (2006). *Data Analysis Using Regression and
  Multilevel/Hierarchical Models*, Chapter 5. Cambridge University Press.
- Berkson, J. (1944). "Application of the Logistic Function to Bio-Assay."
  *JASA*, 39, 357-365.
- The Bangladesh arsenic crisis: Smith, A.H., Lingas, E.O., & Rahman, M.
  (2000). "Contamination of drinking-water by arsenic in Bangladesh."
  *Bulletin of the WHO*, 78(9), 1093-1103.